# mms-1b karaoke finetune

In [ ]:
import json
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
)
from transformers.modeling_outputs import CausalLMOutput

## Load dataset

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")

In [ ]:
max_duration = 120

dataset = dataset.filter(
    lambda x: x.metadata.duration_seconds <= max_duration,
    input_columns="audio",
    num_proc=8,
)

## Prepare

### Feature Extractor

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

### Tokenizer

In [ ]:
vocab = dataset.map(
    lambda batch: {
        "vocab": list(
            set(
                letter
                for sample in batch
                for mora in sample
                for letter in mora["value"]
            )
        )
    },
    input_columns="morae",
    remove_columns=dataset.column_names,
    batched=True,
    batch_size=None,
    keep_in_memory=True,
)["vocab"]

vocab_dict = {v: k for k, v in enumerate(sorted(vocab))}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)

### Processor

In [ ]:
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## Train

In [ ]:
# type: ignore
class Wav2Vec2ForForcedAligner(Wav2Vec2ForCTC):
    """Wav2Vec2ForCTC with frame-level CE loss instead of CTC for forced alignment."""

    def forward(
        self,
        input_values: torch.Tensor | None,
        attention_mask: torch.Tensor | None = None,
        output_attentions: bool | None = None,
        output_hidden_states: bool | None = None,
        return_dict: bool | None = None,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        outputs = super().forward(
            input_values=input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            labels=None,  # skip CTC loss
            **kwargs,
        )
        loss = None
        if labels is not None:
            logits = outputs.logits  # (batch, time, vocab)
            T = min(logits.size(1), labels.size(1))
            ce = F.cross_entropy(
                logits[:, :T].contiguous().view(-1, logits.size(-1)),
                labels[:, :T].contiguous().view(-1),
                ignore_index=-100,
                reduction="none",
            )
            pt = torch.exp(-ce)
            focal = (1 - pt) ** 2 * ce
            mask = labels[:, :T].contiguous().view(-1) != -100
            loss = focal[mask].mean()
        return CausalLMOutput(
            loss=loss,
            logits=outputs.logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
model = Wav2Vec2ForForcedAligner.from_pretrained(
    "facebook/mms-1b",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    # ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,  # type: ignore
    vocab_size=len(processor.tokenizer),  # type: ignore
    ignore_mismatched_sizes=True,
)

# model.freeze_feature_encoder()

# Encoder outputs at 50Hz (20ms/frame); adapter downsamples by adapter_stride when add_adapter=True
adapter_stride = model.config.adapter_stride if model.config.add_adapter else 1
ms_per_frame = 20 * adapter_stride

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    input_values = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],  # type: ignore
    ).input_values[0]
    batch["input_values"] = input_values
    batch["input_length"] = len(input_values)

    # Frame-level labels from morae timings
    num_output_frames = model._get_feat_extract_output_lengths(len(input_values))
    frame_labels = [processor.tokenizer.pad_token_id] * num_output_frames  # type: ignore
    for mora in batch["morae"]:
        token_ids = processor.tokenizer.encode(mora["value"])  # type: ignore
        n_tokens = len(token_ids)
        start_frame = mora["start"] // ms_per_frame
        end_frame = mora["end"] // ms_per_frame
        n_frames = end_frame - start_frame
        if n_tokens > 0 and n_frames > 0:
            for i, token_id in enumerate(token_ids):
                t_start = start_frame + (i * n_frames) // n_tokens
                t_end = start_frame + ((i + 1) * n_frames) // n_tokens
                for f in range(t_start, min(t_end, num_output_frames)):
                    frame_labels[f] = token_id

    batch["labels"] = frame_labels

    return batch


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=8,
    new_fingerprint="mms-1b_prepared_dataset",
)
split = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
@dataclass
class DataCollatorForcedAlignerWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: bool | str = True

    def __call__(
        self, features: list[dict[str, list[int] | torch.Tensor]]
    ) -> dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.pad(
            labels=label_features, padding=self.padding, return_tensors="pt"
        )
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorForcedAlignerWithPadding(processor=processor, padding=True)

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    pred_logits = eval_pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = eval_pred.label_ids

    # Frame-level accuracy (ignoring padding)
    mask = label_ids != -100
    frame_accuracy = ((pred_ids == label_ids) & mask).sum() / mask.sum()  # type: ignore

    # Boundary MAE: compare token transition positions
    pad_id = processor.tokenizer.pad_token_id  # type: ignore
    boundary_errors = []
    for pred_id, label_id in zip(pred_ids, label_ids):
        valid = label_id != -100
        pred_valid, label_valid = pred_id[valid], label_id[valid]
        pred_bounds = [
            i
            for i in range(1, len(pred_valid))
            if pred_valid[i] != pred_valid[i - 1]
            and pred_valid[i] != pad_id
            and pred_valid[i - 1] != pad_id
        ]
        label_bounds = [
            i
            for i in range(1, len(label_valid))
            if label_valid[i] != label_valid[i - 1]
            and label_valid[i] != pad_id
            and label_valid[i - 1] != pad_id
        ]
        for bound in label_bounds:
            if pred_bounds:
                nearest = min(pred_bounds, key=lambda x: abs(x - bound))
                boundary_errors.append(abs(nearest - bound) * ms_per_frame)
            else:
                # No predicted boundaries — penalize with max possible distance
                boundary_errors.append(len(pred_valid) * ms_per_frame)

    boundary_mae_ms = float(np.mean(boundary_errors)) if boundary_errors else 0.0

    return {
        "frame_accuracy": float(frame_accuracy),
        "boundary_mae_ms": boundary_mae_ms,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="mms-1b-ForcedAligner-karaoke-ja-Latn",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=4,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    learning_rate=2e-5,
    warmup_steps=0.05,
    eval_strategy="steps",
    eval_steps=0.1,
    save_steps=0.1,
    save_total_limit=5,
    metric_for_best_model="boundary_mae_ms",
    greater_is_better=False,
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="trackio",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [ ]:
trainer.train()

In [ ]:
processor.save_pretrained("mms-1b-ForcedAligner-karaoke-ja-Latn")
trainer.save_model()